# Math 4512 Extra Credit Notebook
## The 1D Laplacian: D-D case, and setup for D-N / N-D

This notebook focuses on the finite-difference eigenvalue problem for the 1D Laplacian on $[0,L]$.

We will:
- review a few basic NumPy ideas,
- build the **Dirichlet--Dirichlet (D-D)** discrete Laplacian,
- compare its eigenvalues and eigenvectors to the exact continuous eigenpairs,
- study convergence under mesh refinement,
- leave the **D-N** and **N-D** cases for you to complete.

Throughout, we study
$$
-u''(x)=\lambda u(x), \qquad 0<x<L.
$$

## Very short NumPy primer

In this notebook, the main objects are:

- a **NumPy array** such as `x = np.array([1.0, 2.0, 3.0])`, which stores a list of numbers;
- a **matrix**, which is just a 2D NumPy array;
- a **vector of grid values**, such as
  $$
  \mathbf u =
  \begin{bmatrix}
  u_1\\
  u_2\\
  \vdots\\
  u_{N-1}
  \end{bmatrix},
  $$
  which is represented in Python by a 1D array;
- a matrix-vector product such as `A @ v`, which computes $A\mathbf v$.

For example, if
$$
A=
\begin{pmatrix}
2 & -1\\
-1 & 2
\end{pmatrix},
\qquad
\mathbf v=
\begin{pmatrix}
1\\
3
\end{pmatrix},
$$
then `A @ v` computes
$$
\begin{pmatrix}
2 & -1\\
-1 & 2
\end{pmatrix}
\begin{pmatrix}
1\\
3
\end{pmatrix}
=
\begin{pmatrix}
-1\\
5
\end{pmatrix}.
$$

In an eigenvalue problem, we look for a scalar $\mu$ and nonzero vector $\mathbf v$ such that
$$
A\mathbf v=\mu \mathbf v.
$$

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

## Quick array examples

In [ ]:
v = np.array([1.0, 3.0])
A = np.array([[2.0, -1.0],
              [-1.0, 2.0]])

print("v =", v)
print("A =")
print(A)
print("A @ v =", A @ v)

## Continuous eigenpairs for the D-D problem

For
$$
-u''=\lambda u,\qquad 0<x<L,\qquad u(0)=u(L)=0,
$$
the exact eigenpairs are
$$
\lambda_n=\left(\frac{n\pi}{L}\right)^2,
\qquad
u_n(x)=\sin\left(\frac{n\pi x}{L}\right),
\qquad n=1,2,3,\dots
$$

We will compare the discrete matrix eigenvalue problem to these exact formulas.

## Finite differences and the D-D matrix

Let
$$
x_j=jh,\qquad h=\frac{L}{N},\qquad j=0,1,\dots,N.
$$
At interior points, we approximate
$$
-u''(x_j)\approx \frac{-u_{j-1}+2u_j-u_{j+1}}{h^2}.
$$

For the D-D problem, the endpoint values are known:
$$
u_0=0,\qquad u_N=0.
$$
So the unknown vector is just the interior values
$$
\mathbf u=
\begin{bmatrix}
u_1\\
u_2\\
\vdots\\
u_{N-1}
\end{bmatrix}.
$$

This gives the matrix
$$
A_D=\frac1{h^2}
\begin{pmatrix}
2 & -1 & 0 & \cdots & 0\\
-1 & 2 & -1 & \ddots & \vdots\\
0 & -1 & 2 & \ddots & 0\\
\vdots & \ddots & \ddots & \ddots & -1\\
0 & \cdots & 0 & -1 & 2
\end{pmatrix}.
$$

In [ ]:
def dd_matrix(N, L=1.0):
    h = L / N
    m = N - 1
    A = np.zeros((m, m))
    for j in range(m):
        A[j, j] = 2.0 / h**2
        if j > 0:
            A[j, j - 1] = -1.0 / h**2
        if j < m - 1:
            A[j, j + 1] = -1.0 / h**2
    return A

def interior_grid(N, L=1.0):
    h = L / N
    return np.linspace(h, L - h, N - 1)

def exact_dd_eigenvalue(n, L=1.0):
    return (n * np.pi / L)**2

def exact_dd_mode(x, n, L=1.0):
    return np.sin(n * np.pi * x / L)

def normalize_mode(v, exact=None):
    v = np.array(v, dtype=float)
    nv = np.linalg.norm(v)
    if nv > 0:
        v = v / nv
    if exact is not None:
        e = np.array(exact, dtype=float)
        ne = np.linalg.norm(e)
        if ne > 0:
            e = e / ne
        if np.dot(v, e) < 0:
            v = -v
    return v

def grid_mode_error(v_num, v_exact):
    v_num = np.array(v_num, dtype=float)
    v_exact = np.array(v_exact, dtype=float)
    nv = np.linalg.norm(v_num)
    ne = np.linalg.norm(v_exact)
    if nv > 0:
        v_num = v_num / nv
    if ne > 0:
        v_exact = v_exact / ne
    return min(np.linalg.norm(v_num - v_exact), np.linalg.norm(v_num + v_exact))

## First look at the D-D eigenvalues

In [ ]:
L = 1.0
N = 20

A = dd_matrix(N, L=L)
vals, vecs = np.linalg.eigh(A)

print("First five numerical D-D eigenvalues:")
print(vals[:5])

print("\nFirst five exact D-D eigenvalues:")
print([exact_dd_eigenvalue(n, L=L) for n in range(1, 6)])

## Compare discrete eigenvectors to the exact sine eigenfunctions

In [ ]:
N = 40
x = interior_grid(N, L=1.0)
vals, vecs = np.linalg.eigh(dd_matrix(N, L=1.0))

plt.figure(figsize=(8, 5))
for k in [1, 2, 3]:
    v_num = vecs[:, k-1]
    v_exact = exact_dd_mode(x, k, L=1.0)
    v_num = normalize_mode(v_num, v_exact)
    v_exact = normalize_mode(v_exact)

    plt.plot(x, v_num, marker='o', linestyle='-', label=f'numerical mode {k}')
    plt.plot(x, v_exact, linestyle='--', label=f'exact mode {k}')

plt.xlabel('x')
plt.ylabel('mode value')
plt.title('D-D discrete eigenvectors vs exact sine eigenfunctions')
plt.legend()
plt.grid(True)
plt.show()

## Eigenvalue convergence under mesh refinement

In [ ]:
def dd_eigenvalue_errors(mode_numbers=(1,2,3), N_values=(10,20,40,80,160), L=1.0):
    hs = []
    errors = {k: [] for k in mode_numbers}
    for N in N_values:
        h = L / N
        hs.append(h)
        vals, _ = np.linalg.eigh(dd_matrix(N, L=L))
        for k in mode_numbers:
            errors[k].append(abs(vals[k-1] - exact_dd_eigenvalue(k, L=L)))
    return np.array(hs), errors

In [ ]:
N_values = [10, 20, 40, 80, 160]
hs, dd_errors = dd_eigenvalue_errors(mode_numbers=(1,2,3), N_values=N_values, L=1.0)

plt.figure(figsize=(8, 5))
for k in [1, 2, 3]:
    plt.loglog(hs, dd_errors[k], marker='o', label=f'mode {k}')
plt.loglog(hs, hs**2, linestyle='--', label=r'$h^2$ reference')
plt.xlabel('h')
plt.ylabel('eigenvalue error')
plt.title('D-D eigenvalue convergence')
plt.legend()
plt.grid(True, which='both')
plt.show()

## Eigenfunction convergence under mesh refinement

We compare the discrete eigenvector to the sampled exact eigenfunction on the same grid.
Since eigenvectors are only defined up to sign and scaling, we normalize both before computing the error.

In [ ]:
def dd_mode_errors(mode_numbers=(1,2,3), N_values=(10,20,40,80,160), L=1.0):
    hs = []
    errors = {k: [] for k in mode_numbers}
    for N in N_values:
        h = L / N
        hs.append(h)
        x = interior_grid(N, L=L)
        vals, vecs = np.linalg.eigh(dd_matrix(N, L=L))
        for k in mode_numbers:
            v_num = vecs[:, k-1]
            v_exact = exact_dd_mode(x, k, L=L)
            errors[k].append(grid_mode_error(v_num, v_exact))
    return np.array(hs), errors

In [ ]:
hs, dd_mode_errs = dd_mode_errors(mode_numbers=(1,2,3), N_values=N_values, L=1.0)

plt.figure(figsize=(8, 5))
for k in [1, 2, 3]:
    plt.loglog(hs, dd_mode_errs[k], marker='o', label=f'mode {k}')
plt.xlabel('h')
plt.ylabel('grid eigenfunction error')
plt.title('D-D eigenfunction convergence')
plt.legend()
plt.grid(True, which='both')
plt.show()

## Primer for the mixed boundary conditions

Now you will set up the **D-N** and **N-D** cases.

The key idea is:

- the **interior stencil stays the same**,
- only the **boundary treatment changes**.

At interior points, still use
$$
\frac{-u_{j-1}+2u_j-u_{j+1}}{h^2}=\mu u_j.
$$

### D-N case
For
$$
u(0)=0,\qquad u'(L)=0,
$$
you should:
- enforce the left endpoint directly with the Dirichlet condition \(u_0=0\),
- approximate the right derivative with a one-sided difference
  $$
  u'(L)\approx \frac{u_N-u_{N-1}}{h}=0.
  $$

So in matrix language:
- the **left side looks like the D-D case**,
- the **last row must be modified** to reflect the Neumann condition.

### N-D case
For
$$
u'(0)=0,\qquad u(L)=0,
$$
you should:
- approximate the left derivative with a one-sided difference
  $$
  u'(0)\approx \frac{u_1-u_0}{h}=0,
  $$
- enforce the right endpoint directly with the Dirichlet condition \(u_N=0\).

So in matrix language:
- the **first row must be modified** to reflect the Neumann condition,
- the **right side looks like the D-D case**.

### Exact eigenpairs for comparison

For D-N:
$$
\lambda_n=\left(\frac{(n+\tfrac12)\pi}{L}\right)^2,
\qquad
u_n(x)=\sin\left(\frac{(n+\tfrac12)\pi x}{L}\right),
\qquad n=0,1,2,\dots
$$

For N-D:
$$
\lambda_n=\left(\frac{(n+\tfrac12)\pi}{L}\right)^2,
\qquad
u_n(x)=\cos\left(\frac{(n+\tfrac12)\pi x}{L}\right),
\qquad n=0,1,2,\dots
$$

## Student work: implement D-N and N-D

Your task is to:
1. build the discrete matrices,
2. compute their first few eigenvalues and eigenvectors,
3. compare to the exact mixed-case eigenpairs,
4. plot eigenvalue and eigenfunction convergence under mesh refinement.

In [ ]:
# TODO 1:
# Implement the D-N matrix.
#
# Suggested approach:
# - Use interior unknowns only.
# - Start from the D-D matrix.
# - Modify the last row so that it reflects the right Neumann condition
#   u'(L) ≈ (u_N - u_{N-1})/h = 0.

def dn_matrix(N, L=1.0):
    raise NotImplementedError("Students: implement the D-N matrix here.")

In [ ]:
# TODO 2:
# Implement the N-D matrix.
#
# Suggested approach:
# - Use interior unknowns only.
# - Start from the D-D matrix.
# - Modify the first row so that it reflects the left Neumann condition
#   u'(0) ≈ (u_1 - u_0)/h = 0.

def nd_matrix(N, L=1.0):
    raise NotImplementedError("Students: implement the N-D matrix here.")

In [ ]:
# TODO 3:
# Implement the exact mixed-case eigenvalue and eigenfunction formulas.

def exact_dn_eigenvalue(n, L=1.0):
    raise NotImplementedError("Students: implement exact D-N eigenvalues.")

def exact_nd_eigenvalue(n, L=1.0):
    raise NotImplementedError("Students: implement exact N-D eigenvalues.")

def exact_dn_mode(x, n, L=1.0):
    raise NotImplementedError("Students: implement exact D-N eigenfunctions.")

def exact_nd_mode(x, n, L=1.0):
    raise NotImplementedError("Students: implement exact N-D eigenfunctions.")

In [ ]:
# TODO 4:
# Repeat the same type of experiments as in the D-D case:
# - print the first few numerical and exact eigenvalues,
# - plot numerical and exact eigenfunctions together,
# - plot eigenvalue errors vs h,
# - plot eigenfunction errors vs h.

## Questions

1. Why do the D-D eigenvectors look like sampled sine waves?
2. Which matrix rows change when you go from D-D to D-N?
3. Which matrix rows change when you go from D-D to N-D?
4. Why should the mixed cases involve half-wave sine or cosine functions?
5. Which modes are approximated most accurately on coarse grids?